# Week 9: Transformers & LLM Architectures
## Notebook 09: ALiBi: Attention with Linear Biases & Extrapolation

### 1. Conceptual Intuition & Theoretical Background
Standard Transformers fail to extrapolate to sequences longer than those seen during pretraining. When evaluated on sequences twice as long as training context, sinusoidal and learned positional embeddings cause perplexity to explode.

**ALiBi (Press et al., 2021):**
Completely eliminates positional embeddings from the input layer. Instead, it adds a **static, non-learnable linear bias penalty** directly to the attention score matrix:
$$\text{Softmax}\left( \frac{Q K^T}{\sqrt{d_k}} + m \cdot \begin{bmatrix} 0 & -1 & -2 & \dots & -(S-1) \\ 0 & 0 & -1 & \dots & -(S-2) \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & 0 & \dots & 0 \end{bmatrix} \right)$$

Where $m$ is a head-specific geometric slope factor:
$$m = 2^{-\frac{8 \cdot h}{H}}, \quad h \in \{1, 2, \dots, H\}$$
Penalizes attention weights proportional to the distance between query and key tokens, enabling extrapolation to sequences $10\times$ longer than the training horizon.

---

### 2. Scratch Python Implementation: ALiBi Attention Head

In [ ]:
import numpy as np

def get_alibi_slopes(n_heads):
    """Computes geometric slopes m for each attention head."""
    def get_slopes_power_of_2(n):
        start = 2 ** (-(2 ** -(np.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]
    
    if np.log2(n_heads).is_integer():
        return np.array(get_slopes_power_of_2(n_heads))
    else:
        # Interpolate for non-power-of-2 heads
        closest_power = 2 ** int(np.floor(np.log2(n_heads)))
        return np.array(get_slopes_power_of_2(closest_power))

def alibi_attention_forward(Q, K, V, slopes):
    """
    Q, K, V: shape (batch, heads, seq_len, head_dim)
    slopes: shape (heads,)
    """
    B, H, S, d = Q.shape
    d_k = d
    
    # Raw scaled dot-product attention scores: (B, H, S, S)
    scores = np.matmul(Q, K.swapaxes(-2, -1)) / np.sqrt(d_k)
    
    # Construct linear distance bias matrix
    # Distance between position i and j is -(i - j) for causal attention
    indices = np.arange(S)
    distance_matrix = -np.abs(indices[:, np.newaxis] - indices[np.newaxis, :])
    
    # Causal lower-triangular mask
    causal_mask = np.triu(np.full((S, S), -1e9), k=1)
    
    # Broadcast head slopes across distance matrix
    # (H, 1, 1) * (1, S, S) -> (H, S, S)
    alibi_bias = slopes[:, np.newaxis, np.newaxis] * distance_matrix[np.newaxis, :, :]
    
    # Add ALiBi bias and causal mask to attention logits
    scores = scores + alibi_bias[np.newaxis, :, :, :] + causal_mask[np.newaxis, np.newaxis, :, :]
    
    # Softmax
    weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    weights /= np.sum(weights, axis=-1, keepdims=True)
    
    out = np.matmul(weights, V)
    return out, weights

# Simulate 8 Attention Heads on Sequence of Length 16
batch, heads, seq_len, head_dim = 1, 8, 16, 32
Q = np.random.randn(batch, heads, seq_len, head_dim)
K = np.random.randn(batch, heads, seq_len, head_dim)
V = np.random.randn(batch, heads, seq_len, head_dim)

slopes = get_alibi_slopes(heads)
out, attn_weights = alibi_attention_forward(Q, K, V, slopes)

print("--- ALiBi Attention Mechanics ---")
print(f"Calculated Slopes for 8 Heads: {np.round(slopes, 4)}")
print(f"Head 0 Slope (Strong decay):   {slopes[0]:.4f}")
print(f"Head 7 Slope (Weak decay):     {slopes[-1]:.4f}")
print(f"Attention Weights Shape:       {attn_weights.shape}")
print("Linear distance penalty enforces monotonic attention recency without positional embeddings!")

### 3. Step-by-Step Code Walkthrough
1. **Geometric Slopes ($m$):** Assigns steep decay slopes to early heads and gentle slopes to later heads.
2. **Static Distance Matrix:** Penalizes older context without learnable parameters.
3. **Extrapolation:** Models trained on $2,048$ tokens extrapolate cleanly to $8,000+$ tokens without retraining.

---

### 4. Industry Applications & Trade-offs
- **MPT-7B & Falcon Foundation Models:** Adopted ALiBi to support variable context inference.
- **Trade-off:** Pre-defined geometric slopes are non-learnable and may underperform RoPE on complex multi-hop retrieval benchmarks.

---

### 5. Top Interview Questions & Answers
**Q1: Why does ALiBi extrapolate to longer sequence lengths than Rotary Positional Embeddings (RoPE) without fine-tuning?**
> **Answer:** RoPE rotates queries and keys using trigonometric frequencies calibrated to training sequence lengths; longer lengths introduce unseen rotation angles that destabilize attention. ALiBi uses a monotonic linear bias that preserves its inductive recency bias across arbitrary lengths.

**Q2: What is the computational overhead of adding ALiBi biases during attention?**
> **Answer:** Minimal. It adds a precomputed scalar matrix to the attention logits prior to softmax, adding $O(S^2)$ memory bandwidth without matrix multiplication overhead.